In [ ]:
# Bootstrap: jalan di Kaggle / Colab / lokal.
# Kalau '../data' belum ada (runtime baru), clone repo dulu supaya path relatif '../data/...' ketemu.
import os, subprocess
if not os.path.isdir('../data'):
    REPO = 'https://github.com/madajabbar/hybrid-sarima-nbeats_Anc.git'
    ROOT = '/kaggle/working' if os.path.isdir('/kaggle/working') else '/content'
    DEST = os.path.join(ROOT, 'AnC')
    if not os.path.isdir(DEST):
        subprocess.run(['git', 'clone', '-q', '-b', 'main', REPO, DEST], check=True)
    os.chdir(os.path.join(DEST, 'notebooks'))
print('cwd:', os.getcwd(), '| data:', os.path.isfile('../data/transformed_data.csv'))


In [1]:
%pip install -q pmdarima


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 30.3 MB/s eta 0:00:00a 0:00:01


In [5]:
from pmdarima import auto_arima


In [1]:
%pip install -q keras-tuner


In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional, GRU
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping, ModelCheckpoint, LambdaCallback
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from statsmodels.tsa.statespace.sarimax import SARIMAX
from scipy.stats import boxcox
import tensorflow as tf

# Preprocessing data
def preprocess_data(file_path):
    data = pd.read_csv(file_path)
    data['Date'] = pd.to_datetime(data['Date'], format='%d-%m-%y')
    data = data.sort_values(by='Date')
    data.set_index('Date', inplace=True)

    # Apply Box-Cox transformation
    transformed_data = pd.DataFrame(index=data.index)
    lambda_values = {}
    for column in data.columns:
        positive_values = data[column] + abs(data[column].min()) + 1
        transformed_data[column], lambda_values[column] = boxcox(positive_values)

    return data, transformed_data, lambda_values

# Function to create sequences for LSTM
def create_sequences(data, time_steps):
    X, y = [], []
    for i in range(len(data) - time_steps):
        X.append(data[i:i + time_steps])
        y.append(data[i + time_steps])
    return np.array(X), np.array(y)

# Define LSTM model
def build_lstm(input_shape):
    model = Sequential([
        Bidirectional(LSTM(192, return_sequences=True, input_shape=input_shape)),
        Dropout(0.4),
        Bidirectional(LSTM(64, return_sequences=False)),
        Dense(128, activation='relu'),
        Dropout(0.4),
        Dense(1, activation='linear')
    ])
    optimizer = tf.keras.optimizers.AdamW(learning_rate=0.001, weight_decay=1e-5)
    loss = tf.keras.losses.Huber(delta=1.0)
    model.compile(optimizer=optimizer, loss=loss)
    return model

# Evaluate metrics
def evaluate_metrics(actual, forecast):
    rmse = np.sqrt(mean_squared_error(actual, forecast))  # compat: squared= removed in sklearn>=1.6
    mae = mean_absolute_error(actual, forecast)
    mape = np.mean(np.abs((actual - forecast) / actual)) * 100
    return rmse, mae, mape

# File path and data preparation
file_path = '../data/transformed_data.csv'
data, transformed_data, lambda_values = preprocess_data(file_path)

# Split into train and test sets
test_size_days = 180  # Last 6 months for test
train_data = transformed_data.iloc[:-test_size_days]
test_data = transformed_data.iloc[-test_size_days:]
'''
def train_auto_sarima(data, seasonal=True, m=12):
    # Determină automat parametrii SARIMA folosind auto_arima
    auto_sarima_model = auto_arima(
        data,
        start_p=0, start_q=0,
        max_p=5, max_q=5,
        d=None,  # Determină automat diferențierea
        seasonal=seasonal,
        m=m,  # Frecvența sezonală (12 pentru date lunare)
        start_P=0, start_Q=0,
        max_P=2, max_Q=2,
        D=None,  # Determină diferențierea sezonieră
        trace=True,  # Afișează progresul optimizării
        error_action='ignore',  # Ignoră erorile pentru anumite configurații
        suppress_warnings=True,  # Suprimă avertismentele
        stepwise=True  # Utilizează o abordare stepwise pentru eficiență
    )
    return auto_sarima_model

# Train SARIMA and generate predictions
auto_sarima_model = train_auto_sarima(train_data['NO2'], seasonal=True, m=12)
'''
# Afișarea parametrilor aleși de auto_arima
#print(f"Auto ARIMA Model Order: {auto_sarima_model.order}")
#print(f"Auto ARIMA Seasonal Order: {auto_sarima_model.seasonal_order}")

# Conversia modelului auto_arima într-un model SARIMAX pentru ajustare pe datele de antrenament


# Train SARIMA
def train_sarima(data, order, seasonal_order):
    model = SARIMAX(data, order=order, seasonal_order=seasonal_order)
    sarima_fit = model.fit(disp=False)
    predictions = sarima_fit.fittedvalues
    return sarima_fit, predictions

# SARIMA configuration
sarima_order = (2, 1, 1)
seasonal_order = (0, 0, 0, 12)
sarima_model = SARIMAX(
    train_data['NO2'],
    order=sarima_order,
    seasonal_order=seasonal_order
).fit(disp=False)

# Generarea predicțiilor pentru setul de antrenament
sarima_predictions_train = sarima_model.fittedvalues
# Forecast SARIMA on the test set
sarima_predictions_test = sarima_model.forecast(steps=len(test_data))
sarima_predictions = pd.concat(
    [sarima_predictions_train, pd.Series(sarima_predictions_test, index=test_data.index)]
)

# Compute SARIMA residuals
residuals = train_data['NO2'] - sarima_predictions_train
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_residuals = scaler.fit_transform(residuals.values.reshape(-1, 1))

# Generate sequences for LSTM
time_steps = 90
X_train_residuals, y_train_residuals = create_sequences(scaled_residuals, time_steps)
LSTM_STEPS = X_train_residuals.shape[1]  # lungimea secvenței cerută de model (folosită la blocul de orizonturi)

# Check if the test set contains sufficient sequences
test_residuals = test_data['NO2'] - sarima_predictions_test
scaled_test_residuals = scaler.transform(test_residuals.values.reshape(-1, 1))
X_test_residuals, _ = create_sequences(scaled_test_residuals, time_steps)

if len(X_test_residuals) == 0:
    raise ValueError("Not enough data in the test set to create sequences for LSTM. Increase the test set size.")

# Build and train LSTM
model_lstm = build_lstm((X_train_residuals.shape[1], 1))
early_stopping = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-5)

# Checkpoint: /content/drive/... kalau Drive di-mount (survive runtime mati), kalau tidak -> folder notebook ini
# Colab+Drive -> Drive (survive runtime mati) | Kaggle -> /kaggle/working (ikut tersimpan saat commit) | lokal -> folder notebook
if os.path.isdir('/content/drive/MyDrive'):
    CKPT = '/content/drive/MyDrive/AnC_ckpt'
elif os.path.isdir('/kaggle/working'):
    CKPT = '/kaggle/working/AnC_ckpt'
else:
    CKPT = '.'
os.makedirs(f'{CKPT}/checkpoints', exist_ok=True)
print('checkpoint ->', f'{CKPT}/checkpoints')

def fit_resumable(model, tag, X, y, validation_split=0.2, epochs=100, batch_size=64):
    """Fit + simpan bobot tiap epoch. Kalau file ada, lanjut dari epoch terakhir (tak ulang dari nol)."""
    w, st = f'{CKPT}/checkpoints/{tag}.weights.h5', f'{CKPT}/checkpoints/{tag}.epoch'
    start = 0
    if os.path.exists(w):
        if not model.built:  # Keras 3: load_weights butuh model ter-build dulu
            model.build((None,) + tuple(X.shape[1:]))
        model.load_weights(w)
        start = int(open(st).read()) if os.path.exists(st) else 0
        print(f'[{tag}] resume dari epoch {start}/{epochs}')
    if start >= epochs:
        print(f'[{tag}] sudah selesai ({start}/{epochs}) - skip latih')
        return model.history
    cbs = [early_stopping, reduce_lr,
           ModelCheckpoint(w, save_weights_only=True),
           LambdaCallback(on_epoch_end=lambda e, l: open(st, 'w').write(str(e + 1)))]
    return model.fit(X, y, validation_split=validation_split, initial_epoch=start,
                     epochs=epochs, batch_size=batch_size, callbacks=cbs)

t0 = time.time()
fit_resumable(model_lstm, 'model_lstm', X_train_residuals, y_train_residuals)
fit_time_lstm = time.time() - t0                       # ~0 dacă s-a reluat din checkpoint
n_params_lstm = int(model_lstm.count_params())
print(f'BiLSTM: {n_params_lstm:,} parametri | timp fit {fit_time_lstm:.1f} s')

# Predict residuals with LSTM
lstm_predictions = model_lstm.predict(X_test_residuals)
lstm_predictions_rescaled = scaler.inverse_transform(lstm_predictions.reshape(-1, 1))

# Combine SARIMA predictions and LSTM corrections
min_length = min(len(sarima_predictions_test), len(lstm_predictions_rescaled))
sarima_predictions_aligned_trimmed = sarima_predictions_test[-min_length:]
lstm_predictions_rescaled_trimmed = lstm_predictions_rescaled.flatten()[:min_length]


final_predictions_lstm = sarima_predictions_aligned_trimmed.values + lstm_predictions_rescaled_trimmed

# Evaluate metrics
test_actual_values_trimmed = test_data['NO2'][-min_length:]
metrics_lstm = evaluate_metrics(test_actual_values_trimmed, final_predictions_lstm)
metrics_sarima = evaluate_metrics(test_actual_values_trimmed, sarima_predictions_aligned_trimmed)

# Print metrics
print(f"SARIMA Metrics: RMSE={metrics_sarima[0]:.3f}, MAE={metrics_sarima[1]:.3f}, MAPE={metrics_sarima[2]:.2f}%")
print(f"Hybrid (SARIMA + LSTM) Metrics: RMSE={metrics_lstm[0]:.3f}, MAE={metrics_lstm[1]:.3f}, MAPE={metrics_lstm[2]:.2f}%")

# Salvăm metricele de bază (folosite ca baseline de notebook 02)
os.makedirs('../results', exist_ok=True)
pd.DataFrame([
    {'target': 'NO2', 'model': 'SARIMA',        'rmse': metrics_sarima[0], 'mae': metrics_sarima[1], 'mape': metrics_sarima[2],
     'window': 'ultimele 90 zile test', 'n_params': 0,           'fit_time_s': 0.0},
    {'target': 'NO2', 'model': 'SARIMA+BiLSTM', 'rmse': metrics_lstm[0],   'mae': metrics_lstm[1],   'mape': metrics_lstm[2],
     'window': 'ultimele 90 zile test', 'n_params': n_params_lstm, 'fit_time_s': round(fit_time_lstm, 1)},
]).to_csv('../results/01_baseline_metrics.csv', index=False)
print("scris ../results/01_baseline_metrics.csv")

# Plot Actual vs Predictions
plt.figure(figsize=(12, 6))
plt.plot(test_actual_values_trimmed.index, test_actual_values_trimmed, label='Actual Values', marker='o', linestyle='-')
plt.plot(test_actual_values_trimmed.index, sarima_predictions_aligned_trimmed, label='SARIMA Predictions', linestyle='--')
plt.plot(test_actual_values_trimmed.index, final_predictions_lstm, label='Hybrid Predictions (SARIMA + BiLSTM-LSTM)', linestyle='-.')
plt.title("NO2 Predictions: Actual vs SARIMA vs BiLSTM-LSTM")
plt.xlabel("Date")
plt.ylabel("NO2")
plt.legend()
plt.grid(True)
plt.savefig('figno2LSTM.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# Combine SARIMA predictions into a single DataFrame
sarima_predictions = pd.concat([
    pd.Series(sarima_predictions_train, index=train_data.index),
    pd.Series(sarima_predictions_test, index=test_data.index)
], axis=0).rename("NO2").reset_index()
sarima_predictions.columns = ['Date', 'NO2']

# Save SARIMA predictions
sarima_predictions.to_csv('../results/01_baseline_sarima_predictions.csv', index=False)

# Save LSTM predictions
lstm_predictions_rescaled_df = pd.DataFrame({
    'Date': test_data.index[-len(lstm_predictions_rescaled):],
    'NO2': lstm_predictions_rescaled.flatten()
})
lstm_predictions_rescaled_df.to_csv('../results/01_baseline_lstm_predictions.csv', index=False)

# Save Hybrid predictions
hybrid_predictions_df = pd.DataFrame({
    'Date': test_data.index[-len(final_predictions_lstm):],
    'NO2': final_predictions_lstm
})
hybrid_predictions_df.to_csv('../results/01_baseline_hybrid_predictions.csv', index=False)


In [26]:
def build_lstm_tuner(hp):
    model = Sequential()
    model.add(
        Bidirectional(
            LSTM(
                units=hp.Int('units_1', min_value=64, max_value=512, step=64),
                return_sequences=True,
                input_shape=(time_steps, 1)
            )
        )
    )
    model.add(Dropout(hp.Float('dropout_1', min_value=0.1, max_value=0.5, step=0.1)))
    model.add(
        Bidirectional(
            LSTM(
                units=hp.Int('units_2', min_value=64, max_value=256, step=64),
                return_sequences=False
            )
        )
    )
    model.add(Dense(units=hp.Int('dense_units', min_value=64, max_value=256, step=64), activation='relu'))
    model.add(Dropout(hp.Float('dropout_2', min_value=0.1, max_value=0.5, step=0.1)))
    model.add(Dense(1, activation='linear'))
    optimizer = tf.keras.optimizers.AdamW(
        learning_rate=hp.Choice('learning_rate', values=[1e-3, 1e-4, 1e-5]),
        weight_decay=hp.Float('weight_decay', min_value=1e-6, max_value=1e-4, sampling='log')
    )
    loss = tf.keras.losses.Huber(delta=1.0)
    model.compile(optimizer=optimizer, loss=loss)
    return model


In [27]:
from keras_tuner import RandomSearch


In [28]:
tuner = RandomSearch(
    build_lstm_tuner,
    objective='val_loss',
    max_trials=20,
    executions_per_trial=1,
    directory=f'{CKPT}/lstm3_tuning',
    overwrite=False,  # lanjutkan trial yang sudah selesai, bukan mulai ulang
    project_name='sarima_lstm'
)


/opt/conda/lib/python3.10/site-packages/keras/src/layers/rnn/rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [29]:
tuner.search(
    X_train_residuals, y_train_residuals,
    validation_split=0.2,
    epochs=100,
    batch_size=64,
    callbacks=[early_stopping]
)


Trial 20 Complete [00h 00m 29s]
val_loss: 0.0005756666068919003

Best val_loss So Far: 0.000570277392398566
Total elapsed time: 00h 09m 49s


In [30]:
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print(f"""
The optimal number of units in the first layer is {best_hps.get('units_1')},
the second layer is {best_hps.get('units_2')},
the dropout rates are {best_hps.get('dropout_1')} and {best_hps.get('dropout_2')},
and the optimal learning rate is {best_hps.get('learning_rate')}.
""")



The optimal number of units in the first layer is 192,
the second layer is 64,
the dropout rates are 0.4 and 0.4,
and the optimal learning rate is 0.001.



In [13]:
best_model = tuner.hypermodel.build(best_hps)
history = fit_resumable(best_model, 'best_model', X_train_residuals, y_train_residuals)


Epoch 1/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 66ms/step - loss: 0.0204 - val_loss: 0.0036 - learning_rate: 1.0000e-04
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 48ms/step - loss: 0.0046 - val_loss: 0.0023 - learning_rate: 1.0000e-04
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 47ms/step - loss: 0.0038 - val_loss: 0.0022 - learning_rate: 1.0000e-04
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 47ms/step - loss: 0.0040 - val_loss: 0.0022 - learning_rate: 1.0000e-04
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 47ms/step - loss: 0.0037 - val_loss: 0.0021 - learning_rate: 1.0000e-04
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 47ms/step - loss: 0.0040 - val_loss: 0.0022 - learning_rate: 1.0000e-04
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 48ms/step - loss: 0.0036 - val_loss: 0.0021 - learning_rate: 1.0000e-04
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 48ms/step - loss: 0.0038 - val_loss: 0.0022 - learning_rate: 1.0000e-04
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 48ms/step - loss: 0.0034 - val_loss: 0.0021 - 

In [ ]:
# Evaluare pe orizonturi (rolling-origin) - protocol propriu, corect din punct de vedere statistic.
# Fereastră de evaluare extinsă (720 zile) + refit SARIMAX pe split-ul nou, ca prognoza să fie aliniată la date.
feature = 'NO2'
test_size_days = 720
train_data = transformed_data.iloc[:-test_size_days]
test_data = transformed_data.iloc[-test_size_days:]
sarima_model = SARIMAX(train_data[feature], order=sarima_order, seasonal_order=seasonal_order).fit(disp=False)

horizons = [1, 7, 30]
ORIGIN_STEP = 30          # un origin la fiecare 30 de zile
horizon_rows = []

for h in horizons:
    for origin in range(0, len(test_data) - h, ORIGIN_STEP):
        # SARIMA: parametrii deja estimați se aplică pe istoricul extins (apply), fără re-estimare
        hist = pd.concat([train_data[feature], test_data[feature].iloc[:origin]])
        res = sarima_model.apply(hist)
        sarima_h = float(res.forecast(steps=h).iloc[-1])

        # Corecție BiLSTM: secvența celor mai recente reziduuri in-sample -> predicție 1 pas
        resid = (hist - res.fittedvalues).dropna()
        corr = 0.0
        if len(resid) >= LSTM_STEPS:
            seq = scaler.transform(resid.values[-LSTM_STEPS:].reshape(-1, 1)).ravel()
            corr = float(scaler.inverse_transform(
                model_lstm.predict(seq.reshape(1, LSTM_STEPS, 1), verbose=0))[0, 0])

        horizon_rows.append({
            'horizon': h,
            'origin_date': str(test_data.index[origin].date()),
            'target_date': str(test_data.index[origin + h - 1].date()),
            'actual': float(test_data[feature].iloc[origin + h - 1]),
            'sarima': sarima_h,
            'hybrid': sarima_h + corr,
            'lstm_correction': corr,
        })

horizon_df = pd.DataFrame(horizon_rows)


def _horizon_metrics(actual, pred):
    actual, pred = np.asarray(actual, float), np.asarray(pred, float)
    return (float(np.sqrt(np.mean((actual - pred) ** 2))),
            float(np.mean(np.abs(actual - pred))),
            float(np.mean(np.abs((actual - pred) / actual)) * 100))


horizon_metrics = []
print(f"Evaluare rolling-origin ({feature}), un origin la {ORIGIN_STEP} zile")
for h in horizons:
    d = horizon_df[horizon_df.horizon == h]
    for model, col in [('SARIMA', 'sarima'), ('SARIMA+BiLSTM', 'hybrid')]:
        rmse, mae, mape = _horizon_metrics(d.actual, d[col])
        horizon_metrics.append({'target': feature, 'horizon': h, 'model': model,
                                'rmse': rmse, 'mae': mae, 'mape': mape, 'n_origins': len(d)})
        print(f"  h={h:>2} {model:14s} RMSE={rmse:.4f} MAE={mae:.4f} MAPE={mape:.2f}%  (n={len(d)})")

horizon_metrics_df = pd.DataFrame(horizon_metrics)
horizon_metrics_df.to_csv('../results/01_baseline_horizon_metrics.csv', index=False)
horizon_df.to_csv('../results/01_baseline_horizon_predictions.csv', index=False)
print("scris ../results/01_baseline_horizon_metrics.csv + _horizon_predictions.csv")


In [ ]:
from statsmodels.stats.diagnostic import acorr_ljungbox
from scipy.stats import skew, kurtosis
from statsmodels.stats.stattools import durbin_watson


def calculate_residual_metrics(residuals, model_name="Model"):
    """Diagnostică reziduuri: medie, dev.std, skew, kurtosis, Durbin-Watson, Ljung-Box."""
    residuals = np.asarray(residuals, float)
    if len(residuals) < 10:
        print(f"Prea puține date pentru Ljung-Box la {model_name}.")
        return {'Model': model_name, 'Mean': np.mean(residuals), 'Std Dev': np.std(residuals),
                'Skewness': skew(residuals, nan_policy='omit'), 'Kurtosis': kurtosis(residuals, nan_policy='omit'),
                'Durbin-Watson': np.nan, 'Ljung-Box p-value': np.nan}
    try:
        lb_p = float(acorr_ljungbox(residuals, lags=[10], return_df=True)['lb_pvalue'].iloc[0])
    except Exception as e:
        lb_p = np.nan
        print(f"Eroare Ljung-Box ({model_name}): {e}")
    return {'Model': model_name, 'Mean': float(np.mean(residuals)), 'Std Dev': float(np.std(residuals)),
            'Skewness': float(skew(residuals, nan_policy='omit')),
            'Kurtosis': float(kurtosis(residuals, nan_policy='omit')),
            'Durbin-Watson': float(durbin_watson(residuals)), 'Ljung-Box p-value': lb_p}


In [ ]:
# Analiza reziduurilor pe fereastra aliniată din celula de bază (ultimele 90 de zile de test, țintă NO2)
residuals_sarima = pd.Series(test_actual_values_trimmed.values - sarima_predictions_aligned_trimmed.values,
                             index=test_actual_values_trimmed.index, name='SARIMA')
residuals_hybrid = pd.Series(test_actual_values_trimmed.values - final_predictions_lstm,
                             index=test_actual_values_trimmed.index, name='Hybrid')

residuals_metrics = pd.DataFrame([
    calculate_residual_metrics(residuals_sarima.values, "SARIMA"),
    calculate_residual_metrics(residuals_hybrid.values, "SARIMA + BiLSTM"),
])
print(residuals_metrics.to_string(index=False))
residuals_metrics.to_csv('../results/01_baseline_residual_metrics.csv', index=False)

# Serii aliniate (index real, fără NaN) pentru blocul de valori extreme
sarima_predictions_df = pd.Series(sarima_predictions_aligned_trimmed.values,
                                  index=test_actual_values_trimmed.index, name='SARIMA Predictions')
hybrid_predictions_df = pd.Series(final_predictions_lstm,
                                  index=test_actual_values_trimmed.index, name='Hybrid Predictions')
print("scris ../results/01_baseline_residual_metrics.csv")


In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(residuals_sarima.index, residuals_sarima.values, label='Residuals SARIMA', marker='o')
plt.plot(residuals_hybrid.index, residuals_hybrid.values, label='Residuals Hybrid (SARIMA + BiLSTM)', marker='x', linestyle='--')
plt.title("NO2 - Residuals Comparison (SARIMA vs SARIMA + BiLSTM)")
plt.xlabel("Date"); plt.ylabel("Residuals"); plt.legend(); plt.grid(True)
plt.savefig('../results/01_baseline_residuals.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# Performanța pe valori extreme (peste ±2 deviații standard), țintă NO2, fereastră aliniată
mean_val = test_actual_values_trimmed.mean()
std_val = test_actual_values_trimmed.std()
upper_threshold, lower_threshold = mean_val + 2 * std_val, mean_val - 2 * std_val

extreme_actual = test_actual_values_trimmed[(test_actual_values_trimmed > upper_threshold) |
                                            (test_actual_values_trimmed < lower_threshold)]
extreme_sarima = sarima_predictions_df.loc[extreme_actual.index]
extreme_hybrid = hybrid_predictions_df.loc[extreme_actual.index]
print(f"valori extreme: {len(extreme_actual)} zile (prag {lower_threshold:.2f} - {upper_threshold:.2f})")


def calculate_extreme_metrics(actual, prediction, model_name):
    rmse = float(np.sqrt(mean_squared_error(actual, prediction)))
    mae = float(mean_absolute_error(actual, prediction))
    mape = float(np.mean(np.abs((actual - prediction) / actual)) * 100)
    print(f"{model_name} pe valori extreme: RMSE={rmse:.3f} MAE={mae:.3f} MAPE={mape:.2f}%")
    return {'Model': model_name, 'RMSE': rmse, 'MAE': mae, 'MAPE': mape}


extreme_metrics = pd.DataFrame([
    calculate_extreme_metrics(extreme_actual, extreme_sarima, "SARIMA"),
    calculate_extreme_metrics(extreme_actual, extreme_hybrid, "SARIMA + BiLSTM"),
])
print(extreme_metrics.to_string(index=False))
extreme_metrics.to_csv('../results/01_baseline_extreme_metrics.csv', index=False)

plt.figure(figsize=(12, 6))
plt.plot(extreme_actual.index, extreme_actual.values, label='Actual Extreme Values', marker='o', color='blue')
plt.plot(extreme_actual.index, extreme_sarima.values, label='SARIMA Predictions', marker='x', linestyle='--', color='orange')
plt.plot(extreme_actual.index, extreme_hybrid.values, label='Hybrid Predictions (SARIMA + BiLSTM)', marker='^', linestyle='-.', color='green')
plt.title("NO2 - Impact of SARIMA and BiLSTM on Extreme Values")
plt.xlabel("Date"); plt.ylabel("NO2"); plt.legend(); plt.grid(True)
plt.savefig('../results/01_baseline_extreme_values_no2.png', dpi=300, bbox_inches='tight')
plt.show()
